# Phase 0.7 — Stage 2: build the idle-trimmed dataset

Companion to [`Details/phase07_data_verification.md`](Details/phase07_data_verification.md). One job:
take the 50 verified episodes, cut the idle head and tail from each, and push the result to the Hub as a
new dataset that the three Stage 3 training notebooks all read.

**Why this is required, not an optimisation.** Verification measured a **median 71-frame (2.38 s) idle
lead-in**, with **47/50 episodes idle longer than 50 frames**. SmolVLA and π0.5 both have
`chunk_size = 50` fixed by their pretrained action experts, so their entire first chunk from the home
pose would be "stay still" — the self-reinforcing stall Phase 0.6 diagnosed, with no larger-chunk escape
available (ACT got away with `n_action_steps=100`, which is longer than the lead-in; the VLAs cannot
change `chunk_size` without breaking the pretrained action head).

| | |
|---|---|
| Source | `HALDijkstraaa/so101_tray_cylinder_portable_20261003_211912` (50 ep · 35,141 frames · 839 MB) |
| Output | `..._trimmed` — expected **50 ep · 29,462 frames (83.8%)**, episodes 485–743 frames |
| Rule | `dev > 2.0°` from the start/end pose, 5-frame pad — identical to [`scripts/make_trimmed_dataset.py`](scripts/make_trimmed_dataset.py) and the Phase 0.5 notebook |
| Runtime | ~20–40 min, **no GPU needed** (decode + re-encode are CPU-bound) |
| Cost of a disconnect | the whole cell; it is cheap enough to just re-run |

**How to use.** Run Part A once, then B, C, D in order. Part D pushes and then verifies what landed on
the Hub. Do not start Stage 3 until Part D prints `READY`.

> This notebook reads every frame through **PyAV sequentially** rather than through `LeRobotDataset[i]`.
> Random access re-seeks per item (~9 frame/s measured); sequential decode runs ~60× faster. The layout
> this relies on — episodes stored back-to-back inside each video file, zero timestamp gaps — was
> confirmed by the Stage 1 checker, and Part C asserts it again per frame.

# Part A — Setup

## 1 — Runtime

In [ ]:
import multiprocessing, os, shutil, subprocess, sys

print("python     :", sys.version.split()[0])
print("cpu cores  :", multiprocessing.cpu_count())
print("ram        :", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1), "GB")
print("disk free  :", round(shutil.disk_usage("/content").free / 1e9, 1), "GB  (need ~3 GB)")
try:
    print("gpu        :", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                         capture_output=True, text=True).stdout.strip() or "none")
except FileNotFoundError:
    print("gpu        : none")
print("\nNo GPU is needed here - trimming is video decode + re-encode, both CPU-bound.")
print("More cores is what helps; a plain CPU runtime is fine and cheapest.")

## 2 — Install

Pinned to **v0.6.1**, the version that recorded the source dataset, so the trimmed copy is written by the
same code (same `codebase_version`, same AV1 encoder defaults).

In [ ]:
!pip install -q "lerobot[dataset]==0.6.1"
import lerobot, av
print("lerobot", lerobot.__version__, "| PyAV", av.__version__)

> If `import lerobot` fails, restart the runtime (**Runtime > Restart session**) and re-run this cell.

## 3 — Authentication

Pushing creates a **new** dataset repo, which a fine-grained token scoped to existing repos cannot do.
The assertion below catches that in 30 seconds instead of after the 30-minute build.

In [ ]:
import os
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token - add a Colab secret named HF_TOKEN, or run notebook_login()"

os.environ["HF_TOKEN"] = tok
login(token=tok, add_to_git_credential=False)
me = whoami()
print("logged in as:", me["name"])

auth = me.get("auth", {}).get("accessToken", {})
if auth.get("role") == "fineGrained":
    scopes = (auth.get("fineGrained") or {}).get("global", [])
    print("token: fine-grained | global scopes:", scopes)
    assert any("repo.write" in s for s in scopes), (
        "this token cannot CREATE new repos. Make a token with global repo.write "
        "at https://huggingface.co/settings/tokens, or use a classic write token.")
else:
    print("token role:", auth.get("role", "write"))

## 4 — Configuration

In [ ]:
HF_USER = "HALDijkstraaa"
SRC     = f"{HF_USER}/so101_tray_cylinder_portable_20261003_211912"
DST     = f"{SRC.split('/')[-1]}_trimmed"
DST     = f"{HF_USER}/{DST}"
ROOT    = "/content/trimmed"          # where the new dataset is built before it is pushed

# The trim rule, identical to scripts/make_trimmed_dataset.py and the Phase 0.5 notebook: cut the
# lead-in before any joint moves more than THRESH degrees from the start pose, and the tail after the
# arm has settled at its final pose, keeping PAD frames of margin on each side.
THRESH, PAD = 2.0, 5

# What Stage 1 measured on the source, asserted again in Part B so a silent change is caught.
EXPECT_EPISODES, EXPECT_FRAMES = 50, 35141
EXPECT_KEPT = 29462

print(f"source : {SRC}")
print(f"output : {DST}")
print(f"rule   : dev > {THRESH} deg, pad {PAD} frames")

---
# Part B — Measure the trim before building anything

Downloads the source (~839 MB) and computes the trim window for every episode from the **parquet alone** —
no video decoding — so you see exactly what the build will produce before committing to it. The numbers
should match [the Stage 1 report](Details/phase07_data_verification.md) §2.

In [ ]:
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset

src = LeRobotDataset(SRC)
root = Path(src.root)
print(f"source: {src.num_episodes} episodes, {src.num_frames} frames, {src.fps} fps -> {root}")
assert src.num_episodes == EXPECT_EPISODES and src.num_frames == EXPECT_FRAMES, (
    f"source is not what Stage 1 verified ({src.num_episodes} ep / {src.num_frames} frames)")

eps = pd.concat([pd.read_parquet(p)
                 for p in sorted(root.glob("meta/episodes/chunk-*/file-*.parquet"))]
                ).sort_values("episode_index").reset_index(drop=True)
fr = pd.concat([pd.read_parquet(p)
                for p in sorted(root.glob("data/chunk-*/file-*.parquet"))]).sort_values("index")

state  = np.stack(fr["observation.state"].to_numpy()).astype(np.float32)
action = np.stack(fr["action"].to_numpy()).astype(np.float32)
ep_arr = fr["episode_index"].to_numpy()
LEN  = {int(r["episode_index"]): int(r["length"]) for _, r in eps.iterrows()}
FROM = {int(r["episode_index"]): int(r["dataset_from_index"]) for _, r in eps.iterrows()}
TASK = str(eps["tasks"].iloc[0][0])
print("task string:", repr(TASK))

keep = np.zeros(len(fr), bool)
rows = []
for e in range(src.num_episodes):
    idx = np.where(ep_arr == e)[0]
    s = state[idx]
    d0 = np.abs(s - s[0]).max(axis=1)
    d1 = np.abs(s - s[-1]).max(axis=1)
    on  = int(np.argmax(d0 > THRESH)) if (d0 > THRESH).any() else 0
    bk  = np.where(d1 > THRESH)[0]
    off = int(bk[-1]) + 1 if len(bk) else len(s)
    a = max(0, on - PAD)
    b = min(len(s), off + PAD)
    keep[idx[a:b]] = True
    rows.append((e, len(s), a, len(s) - b, b - a))

tab = pd.DataFrame(rows, columns=["episode", "frames", "cut_head", "cut_tail", "kept"])
print(tab.to_string(index=False))
print(f"\nlead-in cut : median {tab.cut_head.median():.0f}  range {tab.cut_head.min()}-{tab.cut_head.max()}")
print(f"tail cut    : median {tab.cut_tail.median():.0f}  range {tab.cut_tail.min()}-{tab.cut_tail.max()}")
print(f"kept        : {tab.kept.sum()}/{len(fr)} frames ({100*tab.kept.sum()/len(fr):.1f}%)")
print(f"episodes    : {tab.kept.min()}-{tab.kept.max()} frames")

assert int(keep.sum()) == int(tab.kept.sum())
assert tab.kept.min() >= 100, "an episode would be shorter than an ACT 100-step chunk"
assert abs(int(keep.sum()) - EXPECT_KEPT) <= 50, (
    f"kept {int(keep.sum())} frames, Stage 1 predicted {EXPECT_KEPT} - investigate before building")
print(f"\nOK - matches the Stage 1 prediction of {EXPECT_KEPT} frames.")

---
# Part C — Build the trimmed dataset

Walks both camera streams sequentially and writes the kept frames. Expect ~20–40 min; progress prints per
episode.

Two things it asserts as it goes, because a silent desync here would produce a dataset whose images and
actions belong to different moments — the worst possible failure, and invisible afterwards:

- every decoded frame's **global index matches the parquet row** it is paired with, for both cameras;
- the **frame count per episode** matches what Part B predicted.

In [ ]:
import shutil, time
import av
from lerobot.configs.video import RGBEncoderConfig

def camera_stream(cam):
    # Yield (global_frame_index, HWC uint8 RGB) for every frame of one camera, in index order.
    # Episodes sit back-to-back inside each video file with zero timestamp gaps (verified in Stage 1),
    # and both the files and the episodes within them are in increasing episode order, so a plain
    # sequential walk emits global indices in increasing order.
    key = f"videos/observation.images.{cam}"
    groups = {}
    for _, r in eps.iterrows():
        groups.setdefault((int(r[f"{key}/chunk_index"]), int(r[f"{key}/file_index"])), []
                          ).append(int(r["episode_index"]))
    for (ch, fi), ep_list in sorted(groups.items()):
        path = root / key / f"chunk-{ch:03d}" / f"file-{fi:03d}.mp4"
        bounds, acc = [], 0
        for e in sorted(ep_list):
            bounds.append((acc, acc + LEN[e], e))
            acc += LEN[e]
        container = av.open(str(path))
        k, bi = 0, 0
        try:
            for frame in container.decode(video=0):
                while bi < len(bounds) and k >= bounds[bi][1]:
                    bi += 1
                if bi >= len(bounds):
                    break
                lo, _, e = bounds[bi]
                yield FROM[e] + (k - lo), frame.to_ndarray(format="rgb24")
                k += 1
        finally:
            container.close()

# Features minus the five lerobot manages itself.
feats = {k: v for k, v in src.features.items()
         if k not in ("index", "episode_index", "frame_index", "timestamp", "task_index")}
img_keys = [k for k in feats if k.startswith("observation.images.")]
cams = [k.split(".")[-1] for k in img_keys]
print("features:", list(feats), "| cameras:", cams)

if Path(ROOT).exists():
    shutil.rmtree(ROOT)

dst = LeRobotDataset.create(
    DST, fps=src.fps, features=feats, root=ROOT, robot_type=src.meta.robot_type, use_videos=True,
    # Match the source encoding exactly so the trimmed copy decodes identically. These are also the
    # library defaults except `preset`, which the source recorded at 12.
    rgb_encoder=RGBEncoderConfig(vcodec="libsvtav1", pix_fmt="yuv420p", g=2, crf=30, preset=12),
)

t0 = time.time()
streams = [camera_stream(c) for c in cams]
cur, written, per_ep = -1, 0, {}

for expected, frames in enumerate(zip(*streams)):
    for cam, (gi, _) in zip(cams, frames):
        assert gi == expected, f"{cam} stream desynced at global index {expected} (got {gi})"
    e = int(ep_arr[expected])
    if e != cur:
        if cur >= 0:
            dst.save_episode()
            dt = time.time() - t0
            print(f"  ep{cur:02d}: {LEN[cur]:4d} -> {per_ep[cur]:4d} frames   "
                  f"[{written} written, {dt/60:.1f} min, {written/max(dt,1e-9):.0f} frame/s]", flush=True)
        cur, per_ep[e] = e, 0
    if keep[expected]:
        frame = {"task": TASK,
                 "observation.state": state[expected],
                 "action": action[expected]}
        for cam, (_, img) in zip(cams, frames):
            frame[f"observation.images.{cam}"] = img
        dst.add_frame(frame)
        written += 1
        per_ep[e] += 1

dst.save_episode()
print(f"  ep{cur:02d}: {LEN[cur]:4d} -> {per_ep[cur]:4d} frames")

dt = time.time() - t0
print(f"\nwrote {written} frames in {dt/60:.1f} min ({written/dt:.0f} frame/s)")
for e in range(src.num_episodes):
    assert per_ep[e] == int(tab.kept[tab.episode == e].iloc[0]), f"episode {e} length mismatch"
assert written == int(tab.kept.sum()), f"wrote {written}, expected {tab.kept.sum()}"
print("every episode matches the Part B prediction.")

---
# Part D — Verify, then push

Checks the local build before anything is uploaded, pushes it, then checks what actually landed on the
Hub. `push_to_hub(tag_version=True)` is what creates the **codebase-version git tag** — lerobot refuses
to load a dataset without it, and a plain `hf upload` does not create one
(see [`scripts/push_dataset.sh`](scripts/push_dataset.sh)).

In [ ]:
import json
chk = LeRobotDataset(DST, root=ROOT)
print(f"local: {chk.num_episodes} episodes, {chk.num_frames} frames, {chk.fps} fps")
assert chk.num_episodes == src.num_episodes, "episode count changed"
assert chk.num_frames == written, "frame count changed"

info = json.loads((Path(ROOT) / "meta" / "info.json").read_text())
print("codebase_version:", info["codebase_version"], "| robot_type:", info["robot_type"])
print("cameras:", [k for k in info["features"] if k.startswith("observation.images")])

# The point of the exercise: the lead-in must be gone.
nf = pd.concat([pd.read_parquet(p)
                for p in sorted(Path(ROOT).glob("data/chunk-*/file-*.parquet"))]).sort_values("index")
ns = np.stack(nf["observation.state"].to_numpy()).astype(np.float32)
ne = nf["episode_index"].to_numpy()
on = []
for e in range(chk.num_episodes):
    s = ns[ne == e]
    d0 = np.abs(s - s[0]).max(axis=1)
    on.append(int(np.argmax(d0 > THRESH)) if (d0 > THRESH).any() else 0)
on = np.array(on)
print(f"\nlead-in after trimming: median {np.median(on):.0f} frames, max {on.max()} "
      f"(was median 71, max 125)")
assert on.max() <= PAD + 2, f"an episode still has a {on.max()}-frame lead-in"

item = chk[0]
print("sample item:", {k: tuple(v.shape) for k, v in item.items() if hasattr(v, "shape")})
print("task:", repr(item["task"]))
print("\nlocal build OK")

In [ ]:
dst.push_to_hub(private=True, tag_version=True)
print(f"pushed -> https://huggingface.co/datasets/{DST}")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

files = api.list_repo_files(DST, repo_type="dataset")
tags = [t.name for t in api.list_repo_refs(DST, repo_type="dataset").tags]
ver = json.loads((Path(ROOT) / "meta" / "info.json").read_text())["codebase_version"]

checks = [
    ("meta/info.json uploaded",      "meta/info.json" in files),
    ("meta/stats.json uploaded",     "meta/stats.json" in files),
    ("data parquet uploaded",        any(f.startswith("data/") for f in files)),
    ("videos uploaded",              sum(f.startswith("videos/") for f in files) > 0),
    (f"codebase-version tag {ver}",  ver in tags),
]
for name, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

# The decisive test: load it the way lerobot-train will, straight from the Hub, and pull an action
# chunk of each size the three policies need.
import tempfile
hub = LeRobotDataset(DST)
print(f"\nreloaded from the Hub: {hub.num_episodes} episodes, {hub.num_frames} frames")
ok_reload = hub.num_episodes == src.num_episodes and hub.num_frames == written

for name, chunk in (("ACT (100)", 100), ("SmolVLA / pi0.5 (50)", 50)):
    probe = LeRobotDataset(DST, delta_timestamps={"action": [i / hub.fps for i in range(chunk)]},
                           episodes=[0, 25, 49])
    a = probe[0]["action"]
    good = tuple(a.shape) == (chunk, 6)
    print(f"  {'PASS' if good else 'FAIL'}  action chunk {name}: shape {tuple(a.shape)}")
    ok_reload &= good

READY = all(ok for _, ok in checks) and ok_reload
print(f"\n{'=' * 70}\n  DATASET: {DST}\n  READY: {READY}\n{'=' * 70}")
if READY:
    print("\nStage 3 can start. Use this repo id in all three training notebooks:")
    print(f"    DATASET = \"{DST}\"")
else:
    raise RuntimeError("verification failed - do not start Stage 3")

---
# Part E — If something goes wrong

**The build died partway.** Nothing was pushed, so just re-run Part C — it deletes `/content/trimmed`
first. If Colab ran out of disk, clear the HF cache with `!rm -rf ~/.cache/huggingface/lerobot` and
re-run from Part B.

**`push_to_hub` failed but the build is fine.** Re-run only the push cell; it is idempotent. If the repo
was created but the tag is missing, add it by hand:

```python
from huggingface_hub import HfApi
HfApi().create_tag(DST, tag="v3.0", repo_type="dataset")
```

**You want to start over on the Hub.**

```python
from huggingface_hub import HfApi
HfApi().delete_repo(DST, repo_type="dataset")   # irreversible
```

**A stream-desync assertion fired.** Stop — do not push. It means the video files are not laid out the
way the metadata says, and a dataset built from them would pair images with the wrong actions. Re-run the
Stage 1 checker against the source (`python scripts/verify_phase07_dataset.py`) and look at section 3.

**Changing the trim rule.** `THRESH` and `PAD` are in the Part A config. Anything other than
`2.0 / 5` makes this dataset non-comparable with the Phase 0.6 F3 trimmed runs, so note it in the
dataset card if you change it.

---

### After this notebook

Record in [`Details/phase07_portable_rig.md`](Details/phase07_portable_rig.md) §8 that the trimmed
dataset exists, and write the dataset card — it should say: idle-trimmed copy of
`..._20261003_211912`, new 2026-10-03 calibration frame, bird's-eye `top`, **not** compatible with
`so101_toolkit_cylinder_20260917_165544`.